# K-Nearest Neighbours (KNN) Analysis for Heart Disease Prediction
**Machine Learning Assignment — Member 5**  
**Algorithm: K-Nearest Neighbours Classifier**

---

## 1. Problem Definition

Predict whether a patient has heart disease using the Cleveland dataset. KNN classifies a new sample by majority vote of its **k** nearest training neighbours using Euclidean distance.

- **0** = no heart disease
- **1** = heart disease present

## 2. Dataset Description

| Field | Value |
|---|---|
| Source | `Data_set/processed.cleveland.data` |
| Features | 13 clinical attributes |
| Samples | 303 instances |

## 3. Imports & Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

ROOT_DIR = os.path.abspath(os.path.join('..'))
sys.path.append(ROOT_DIR)

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

from src.data_preprocessing import load_data, preprocess_data, split_features_target
from src.evaluation import save_metrics

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('All imports successful')

## 4. Data Loading & Preprocessing

In [ ]:
DATA_PATH = os.path.join('..', 'Data_set', 'processed.cleveland.data')
raw_df    = load_data(DATA_PATH)
clean_df  = preprocess_data(raw_df)

print(f'Raw shape   : {raw_df.shape}')
print(f'Clean shape : {clean_df.shape}')
print(f'Missing values after preprocessing: {clean_df.isnull().sum().sum()}')
print(f'\nTarget distribution:')
print(clean_df['target'].value_counts())
clean_df.head()

### 4.1 Feature Distributions by Class

In [ ]:
features = [c for c in clean_df.columns if c != 'target']
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(features):
    for label, color in zip([0, 1], ['#2196F3', '#F44336']):
        axes[i].hist(
            clean_df[clean_df['target'] == label][feat],
            bins=15, alpha=0.6, color=color,
            label='No Disease' if label == 0 else 'Disease'
        )
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)

for j in range(len(features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions by Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
RESULTS_DIR = os.path.join('..', 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, 'knn_feature_distributions.png'), bbox_inches='tight', dpi=100)
plt.show()

## 5. Why K-Nearest Neighbours?

| Advantage | Explanation |
|---|---|
| No training phase | KNN memorises training data; predictions happen at query time |
| Non-parametric | Makes no assumptions about data distribution |
| Intuitive | Similar patients should have similar diagnoses |
| Multi-class ready | Extends naturally to any number of classes |

> **Important:** KNN is distance-based — `StandardScaler` is mandatory to prevent features with large ranges dominating.

## 6. Feature Scaling & Train/Test Split

In [ ]:
X, y = split_features_target(clean_df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
X_all_scaled   = scaler.fit_transform(X)

print(f'Train : {X_train.shape[0]} samples | Test : {X_test.shape[0]} samples')
print('Features scaled with StandardScaler')

## 7. Optimal K Selection — Elbow Method

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

k_range, cv_acc_list, cv_std_list = range(1, 31), [], []
for k in k_range:
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=k, metric='euclidean'),
        X_all_scaled, y, cv=cv, scoring='accuracy'
    )
    cv_acc_list.append(scores.mean())
    cv_std_list.append(scores.std())

best_k   = list(k_range)[int(np.argmax(cv_acc_list))]
best_acc = max(cv_acc_list)
print(f'Best k = {best_k}  →  CV Accuracy = {best_acc:.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(k_range), cv_acc_list, color='#1976D2', linewidth=2.2,
        marker='o', markersize=5, label='CV Accuracy')
ax.fill_between(
    list(k_range),
    [a - s for a, s in zip(cv_acc_list, cv_std_list)],
    [a + s for a, s in zip(cv_acc_list, cv_std_list)],
    alpha=0.15, color='#1976D2', label='±1 Std Dev'
)
ax.axvline(best_k, color='#F44336', linestyle='--', linewidth=2,
           label=f'Best k = {best_k} ({best_acc:.3f})')
ax.set_title('Elbow Method — K vs 5-Fold CV Accuracy', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of Neighbours (k)', fontsize=11)
ax.set_ylabel('CV Accuracy', fontsize=11)
ax.set_xticks(list(k_range))
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'knn_elbow_k.png'), dpi=120)
plt.show()

## 8. Model Training

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k, metric='euclidean', weights='uniform')
knn.fit(X_train_scaled, y_train)

print(f'KNN model trained  |  k={knn.n_neighbors}  |  metric={knn.metric}  |  weights={knn.weights}')

## 9. Model Evaluation

In [ ]:
y_pred       = knn.predict(X_test_scaled)
y_pred_proba = knn.predict_proba(X_test_scaled)[:, 1]

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_pred_proba)
cm   = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=' * 52)
print('   K-NEAREST NEIGHBOURS — EVALUATION METRICS')
print('=' * 52)
print(f'  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print(f'  ROC AUC   : {auc:.4f}')
print(f'  Sensitivity: {tp/(tp+fn):.4f}')
print(f'  Specificity: {tn/(tn+fp):.4f}')
print('=' * 52)

In [ ]:
print('CLASSIFICATION REPORT:')
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

print('CONFUSION MATRIX:')
print(f'  TN={tn}  FP={fp}')
print(f'  FN={fn}  TP={tp}')

## 10. Results Visualisation

In [ ]:
fig = plt.figure(figsize=(14, 5))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

# Confusion Matrix
ax1 = fig.add_subplot(gs[0])
sns.heatmap(
    np.array([[tn, fp], [fn, tp]]),
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted\nNo Disease', 'Predicted\nDisease'],
    yticklabels=['Actual\nNo Disease', 'Actual\nDisease'],
    linewidths=1, linecolor='white', ax=ax1,
    annot_kws={'size': 14, 'weight': 'bold'}
)
ax1.set_title(f'Confusion Matrix (k={best_k})', fontsize=13, fontweight='bold')

# ROC Curve
ax2 = fig.add_subplot(gs[1])
fpr_c, tpr_c, _ = roc_curve(y_test, y_pred_proba)
ax2.plot(fpr_c, tpr_c, color='#1976D2', linewidth=2.5,
         label=f'KNN k={best_k} (AUC={auc:.4f})')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1)
ax2.fill_between(fpr_c, tpr_c, alpha=0.1, color='#1976D2')
ax2.set_title('ROC Curve', fontsize=13, fontweight='bold')
ax2.set_xlabel('False Positive Rate')
ax2.set_ylabel('True Positive Rate')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.spines[['top', 'right']].set_visible(False)

plt.suptitle('KNN — Evaluation Results', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(RESULTS_DIR, 'knn_evaluation.png'), bbox_inches='tight', dpi=120)
plt.show()
print('Evaluation plots saved')

## 11. Distance Metric Comparison

In [ ]:
metric_results = []
for met in ['euclidean', 'manhattan', 'chebyshev']:
    scores = cross_val_score(
        KNeighborsClassifier(n_neighbors=best_k, metric=met),
        X_all_scaled, y, cv=cv, scoring='accuracy'
    )
    metric_results.append({'Metric': met.capitalize(),
                            'Mean CV Acc': scores.mean(),
                            'Std': scores.std()})

metric_df = pd.DataFrame(metric_results).sort_values('Mean CV Acc', ascending=False)
print(metric_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(metric_df['Metric'], metric_df['Mean CV Acc'],
              yerr=metric_df['Std'], color=['#1565C0', '#43A047', '#FB8C00'],
              width=0.45, edgecolor='white', capsize=5)
for bar, val in zip(bars, metric_df['Mean CV Acc']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0.6, 1.0)
ax.set_title('Distance Metric Comparison', fontsize=12, fontweight='bold')
ax.set_ylabel('5-Fold CV Accuracy')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'knn_metric_comparison.png'), dpi=120)
plt.show()

## 12. Cross-Validation

In [ ]:
cv_scores = cross_val_score(knn, X_all_scaled, y, cv=cv, scoring='accuracy')
cv_f1     = cross_val_score(knn, X_all_scaled, y, cv=cv, scoring='f1')
cv_auc    = cross_val_score(knn, X_all_scaled, y, cv=cv, scoring='roc_auc')

print('5-Fold Cross-Validation:')
print(f'  Accuracy : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'  F1-Score : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  ROC AUC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars  = ax.bar(folds, cv_scores, color='#1976D2', width=0.5, edgecolor='white')
ax.axhline(cv_scores.mean(), color='#F44336', linewidth=2, linestyle='--',
           label=f'Mean = {cv_scores.mean():.4f}')
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_title(f'5-Fold CV Accuracy — KNN (k={best_k})', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'knn_cross_validation.png'), dpi=120)
plt.show()

## 13. Save Results

In [ ]:
knn_metrics = {
    'accuracy': acc, 'precision': prec, 'recall': rec,
    'f1_score': f1, 'roc_auc': auc, 'confusion_matrix': cm,
    'classification_report': classification_report(
        y_test, y_pred, target_names=['No Disease', 'Disease']
    )
}
metrics_path = os.path.join(RESULTS_DIR, 'knn_metrics.txt')
save_metrics(knn_metrics, metrics_path)
print(f'Metrics saved to: {metrics_path}')

## 14. Interpretation of Results

**Model Performance:**
- KNN achieves competitive performance without any explicit training phase
- The Elbow Method identified the optimal k, balancing bias and variance
- Small k → low bias, high variance (overfitting); large k → high bias, low variance (underfitting)

**Distance Metrics:**
- Euclidean distance performs well after standardisation
- Manhattan distance is more robust to outliers in individual feature dimensions

**Clinical Context:**
- High Recall (Sensitivity) is critical — minimises missed heart disease diagnoses
- Lower threshold on `predict_proba` can trade precision for higher sensitivity

## 15. Critical Analysis

### Strengths
- No training phase — trivial to update as new data arrives
- Non-parametric — no distributional assumptions
- Intuitive and interpretable reasoning

### Limitations
- Slow at prediction time — O(n) distance computation
- Sensitive to feature scale and irrelevant features
- Suffers from the curse of dimensionality
- No built-in feature importance

## 16. Future Improvements
- Use `weights='distance'` so closer neighbours have more influence
- Apply PCA before KNN to reduce dimensionality
- Use Ball Tree or KD-Tree for faster query-time predictions
- GridSearchCV to jointly optimise k, metric, and weights
- SMOTE to handle mild class imbalance

## 17. Individual Contribution (Member 5)

- Implemented KNN with `sklearn.neighbors.KNeighborsClassifier`
- Applied StandardScaler (fit on training data only)
- Used Elbow Method to select optimal k (k=1..30)
- Compared Euclidean, Manhattan, Chebyshev distance metrics
- Generated: Accuracy, Precision, Recall, F1, ROC-AUC, Sensitivity, Specificity
- Visualisations: Elbow curve, Confusion Matrix, ROC Curve, CV chart
- Saved all results to `results/` directory

**Files created:** `notebooks/05_knn_analysis.ipynb`, `results/knn_metrics.txt`, `results/knn_evaluation.png`, `results/knn_elbow_k.png`, `results/knn_cross_validation.png`, `results/knn_metric_comparison.png`